In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="09-similar-listing",
    notebook_name="03_ranking_and_serving.ipynb"
)

# Similar Listings: Ranking and Serving

## The Big Idea (For a 12-Year-Old)

Imagine you run a lemonade stand directory with 5 million stands listed. When someone is looking at a lemonade stand in Miami, you want to show them OTHER great stands nearby. But you cannot just show the closest ones -- you need to think about:

- **Diversity**: Do not show 10 stands all owned by the same person!
- **Fairness**: New stands deserve a chance to be seen too.
- **Business rules**: If a stand is closed for winter, do not recommend it.
- **Speed**: Show recommendations instantly, not after 10 seconds.

This notebook covers everything that happens AFTER the embedding model produces candidate listings -- the ranking, re-ranking, and serving infrastructure.

---

## Staff-Level Technical Summary

We cover:
- **Re-ranking strategies**: Diversity, price filters, geographic constraints, business rules
- **Position bias**: Why top-ranked listings get unfair clicks, and how to fix it
- **Serving architecture**: Airbnb-style three-pipeline system
- **Cold-start handling**: Strategies for new listings with no interaction data
- **Online learning and model updates**: Daily retraining cycle
- **A/B testing**: Experimentation framework for listing recommendations

## 1. Re-Ranking Strategies

### Why Re-Ranking Matters

The embedding model gives us listings sorted by **embedding similarity** -- but raw similarity is not the same as the best user experience. We need a re-ranking layer to apply common-sense rules.

**Simple explanation**: The embedding model is like a matchmaker who knows which houses are "similar." But the re-ranking service is the wise friend who says: "Yes, those are all similar, but let me reorganize the list because you probably do not want 5 listings from the same host, and this one is closed next month."

### Re-Ranking Rules

| Rule | What It Does | Why It Matters |
|------|-------------|----------------|
| **Price filter** | Remove listings above user's price limit | Users set price filters -- respect them |
| **Availability filter** | Remove listings booked for user's dates | No point showing unavailable listings |
| **Geographic constraint** | Only show listings in same city/region | Miami user does not want Alaska cabins |
| **Host diversity** | Max 2 listings per host | Prevents one host from dominating |
| **Type diversity** | Mix different listing types | Show apartments AND houses AND studios |
| **Freshness boost** | Slight boost to newly listed properties | Helps new listings get visibility |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

np.random.seed(42)

# === Re-Ranking Engine for Similar Listings ===

class ListingReRanker:
    """
    Re-ranking service for similar listing recommendations.
    
    12-year-old version:
    The embedding model found 50 houses similar to the one you
    are looking at. But before showing them, we need to:
    1. Remove houses you cannot afford
    2. Remove houses in other cities
    3. Make sure we show different TYPES of houses
    4. Give new houses a chance to be seen
    
    Staff-level detail:
    Re-ranking is a deterministic rule engine (not ML).
    Rules are ordered by priority: filters first, then boosts.
    Rule weights are tuned via A/B testing.
    """
    def __init__(self):
        self.filter_log = []
    
    def apply_price_filter(self, listings, max_price):
        """Remove listings above the user's price filter."""
        if max_price is None:
            return listings
        before = len(listings)
        listings = [l for l in listings if l['price'] <= max_price]
        self.filter_log.append(f'Price filter (max ${max_price}): {before} -> {len(listings)}')
        return listings
    
    def apply_city_filter(self, listings, target_city):
        """Only keep listings in the same city."""
        before = len(listings)
        listings = [l for l in listings if l['city'] == target_city]
        self.filter_log.append(f'City filter ({target_city}): {before} -> {len(listings)}')
        return listings
    
    def apply_availability_filter(self, listings):
        """Remove unavailable listings."""
        before = len(listings)
        listings = [l for l in listings if l.get('available', True)]
        self.filter_log.append(f'Availability filter: {before} -> {len(listings)}')
        return listings
    
    def enforce_host_diversity(self, listings, max_per_host=2):
        """Limit listings from the same host."""
        host_counts = {}
        diverse = []
        for l in listings:
            host = l.get('host_id', 'unknown')
            host_counts[host] = host_counts.get(host, 0) + 1
            if host_counts[host] <= max_per_host:
                diverse.append(l)
        self.filter_log.append(f'Host diversity (max {max_per_host}): {len(listings)} -> {len(diverse)}')
        return diverse
    
    def enforce_type_diversity(self, listings, max_per_type=3):
        """Mix different listing types."""
        type_counts = {}
        diverse = []
        for l in listings:
            ltype = l.get('listing_type', 'unknown')
            type_counts[ltype] = type_counts.get(ltype, 0) + 1
            if type_counts[ltype] <= max_per_type:
                diverse.append(l)
        self.filter_log.append(f'Type diversity (max {max_per_type}): {len(listings)} -> {len(diverse)}')
        return diverse
    
    def apply_freshness_boost(self, listings, boost_weight=0.05):
        """Boost newly listed properties."""
        for l in listings:
            days_listed = l.get('days_listed', 100)
            if days_listed < 30:
                l['score'] += boost_weight * (1 - days_listed / 30)
        return sorted(listings, key=lambda x: -x['score'])
    
    def rerank(self, listings, target_city, max_price=None, top_k=10):
        """Apply all re-ranking rules in order."""
        self.filter_log = []
        
        # Filters (remove items)
        listings = self.apply_availability_filter(listings)
        listings = self.apply_city_filter(listings, target_city)
        listings = self.apply_price_filter(listings, max_price)
        
        # Diversity (reorder/filter)
        listings = self.enforce_host_diversity(listings)
        listings = self.enforce_type_diversity(listings)
        
        # Boosts (adjust scores)
        listings = self.apply_freshness_boost(listings)
        
        return listings[:top_k]


# Generate mock candidate listings
hosts = ['host_A', 'host_A', 'host_A', 'host_B', 'host_B', 'host_C',
         'host_D', 'host_E', 'host_F', 'host_A', 'host_G', 'host_H',
         'host_I', 'host_J', 'host_K', 'host_B', 'host_L', 'host_M',
         'host_N', 'host_O']

cities = ['Miami'] * 14 + ['NYC'] * 3 + ['LA'] * 3
types = ['Beach House', 'Beach Condo', 'Beach House', 'City Apt', 'Penthouse',
         'Beach House', 'Beach Condo', 'Studio', 'Villa', 'Beach House',
         'Beach Condo', 'Studio', 'Villa', 'Beach House', 'City Apt',
         'Penthouse', 'Beach House', 'Studio', 'Villa', 'Beach Condo']

candidates = [
    {
        'listing_id': f'L{i}',
        'score': 1.0 - i * 0.04 + np.random.uniform(-0.02, 0.02),
        'host_id': hosts[i],
        'city': cities[i],
        'listing_type': types[i],
        'price': np.random.randint(100, 500),
        'available': i != 5,  # L5 is unavailable
        'days_listed': np.random.randint(1, 200),
    }
    for i in range(20)
]

reranker = ListingReRanker()

print('=== Before Re-Ranking (Top 10 by Raw Similarity) ===')
print(f'{"Rank":<5} {"ID":<5} {"Score":<8} {"Host":<10} {"City":<8} {"Type":<15} {"Price":<8} {"Avail":<6} {"Days":<5}')
print('-' * 75)
for i, l in enumerate(candidates[:10], 1):
    avail = 'Yes' if l['available'] else 'NO'
    print(f'  {i:<3} {l["listing_id"]:<5} {l["score"]:<8.3f} {l["host_id"]:<10} {l["city"]:<8} {l["listing_type"]:<15} ${l["price"]:<7} {avail:<6} {l["days_listed"]}')

print('\n=== Applying Re-Ranking ===')
reranked = reranker.rerank(candidates, target_city='Miami', max_price=350, top_k=8)

for log in reranker.filter_log:
    print(f'  {log}')

print(f'\n=== After Re-Ranking (Final 8 Results) ===')
print(f'{"Rank":<5} {"ID":<5} {"Score":<8} {"Host":<10} {"City":<8} {"Type":<15} {"Price":<8} {"Days":<5}')
print('-' * 65)
for i, l in enumerate(reranked, 1):
    print(f'  {i:<3} {l["listing_id"]:<5} {l["score"]:<8.3f} {l["host_id"]:<10} {l["city"]:<8} {l["listing_type"]:<15} ${l["price"]:<7} {l["days_listed"]}')

## 2. Position Bias: The Hidden Enemy

### The Front-of-the-Store Problem

**Simple explanation**: Imagine a bookstore where books at eye level get picked up 10x more than books on the bottom shelf. It is not because they are better -- it is just because they are easier to see! This is **position bias**: items shown in position 1 get more clicks than items in position 5, regardless of actual quality.

**Staff-level detail**: Position bias contaminates our training data. If we train on click data, the model learns that "being in position 1" correlates with clicks. It is learning a spurious correlation, not true relevance.

### Two Solutions

1. **Inverse Propensity Weighting (IPW)**: Weight each click by 1/P(click|position). Clicks on position 5 listings count MORE because they are "harder" clicks.

2. **Position-aware training**: Include position as a feature during training, then set position to 0 (or a neutral value) during inference.

In [ ]:
# === Position Bias Demonstration ===

# Simulate click data with position bias
positions = np.arange(1, 11)
# True relevance CTR (what we want to learn)
true_relevance = np.array([0.5, 0.6, 0.3, 0.8, 0.4, 0.7, 0.2, 0.5, 0.6, 0.3])
# Position bias multiplier (exponential decay)
position_bias = np.exp(-0.3 * (positions - 1))
# Observed CTR = true_relevance * position_bias
observed_ctr = true_relevance * position_bias

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Position bias alone
axes[0].bar(positions, position_bias, color='#FF7043', edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('Position', fontsize=11)
axes[0].set_ylabel('Bias Multiplier', fontsize=11)
axes[0].set_title('Position Bias\n(Higher position = more clicks)', fontsize=12, fontweight='bold')
axes[0].set_xticks(positions)
axes[0].grid(True, alpha=0.3, axis='y')

# True relevance
axes[1].bar(positions, true_relevance, color='#66BB6A', edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('Position', fontsize=11)
axes[1].set_ylabel('True Relevance', fontsize=11)
axes[1].set_title('True Relevance\n(What we WANT to learn)', fontsize=12, fontweight='bold')
axes[1].set_xticks(positions)
axes[1].grid(True, alpha=0.3, axis='y')

# Observed CTR (contaminated)
axes[2].bar(positions, observed_ctr, color='#42A5F5', edgecolor='black', linewidth=0.5)
axes[2].set_xlabel('Position', fontsize=11)
axes[2].set_ylabel('Observed CTR', fontsize=11)
axes[2].set_title('Observed CTR\n(True relevance x Position bias)', fontsize=12, fontweight='bold')
axes[2].set_xticks(positions)
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Position Bias Contaminates Training Data', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Demonstrate IPW correction
print('=== Position Bias Correction with IPW ===')
print(f'\n{"Position":<10} {"Obs CTR":<10} {"Bias":<10} {"IPW Weight":<12} {"Corrected":<10} {"True Rel":<10}')
print('-' * 65)
for i in range(10):
    ipw_weight = 1 / position_bias[i]
    corrected = observed_ctr[i] * ipw_weight
    print(f'  {positions[i]:<8} {observed_ctr[i]:<10.3f} {position_bias[i]:<10.3f} '
          f'{ipw_weight:<12.3f} {corrected:<10.3f} {true_relevance[i]:<10.3f}')

print(f'\nIPW corrected values match true relevance!')
print(f'Key insight: Clicks on lower positions are weighted MORE because')
print(f'users had to actively scroll down to find and click them.')

## 3. Serving Architecture (Airbnb-Style)

### Three Pipelines Working Together

**Simple explanation**: Think of it as a factory with three departments:
1. **Training department** (runs every day): Learns new patterns from yesterday's data
2. **Indexing department** (runs after training): Computes and stores every listing's "fingerprint" (embedding)
3. **Serving department** (runs in real-time): When a user views a listing, instantly finds similar ones

```
TRAINING PIPELINE          INDEXING PIPELINE          PREDICTION PIPELINE
+------------------+      +------------------+      +----------------------+
| Daily retraining |----->| Pre-compute all  |----->| 1. Embedding Fetcher |
| on new sessions  |      | listing embeddings|     |    (lookup or cold   |
| and interactions |      | Store in ANN index|     |     start heuristic) |
+------------------+      +------------------+      | 2. Nearest Neighbor  |
                                                    |    (ANN: FAISS/ScaNN)|
                                                    | 3. Re-Ranking        |
                                                    |    (filters, rules)  |
                                                    +----------------------+
```

In [ ]:
# === Full Serving Architecture Diagram ===

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')

def draw_box(ax, x, y, w, h, text, color, edge='#333', fontsize=9):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor=edge, linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=fontsize, fontweight='bold')

def draw_arrow(ax, x1, y1, x2, y2, color='#333', style='->'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle=style, color=color, lw=2))

# Title
ax.text(8, 9.5, 'Similar Listing Serving Architecture', ha='center',
        fontsize=16, fontweight='bold')

# --- Training Pipeline (left) ---
ax.text(2.5, 9, 'TRAINING PIPELINE (Daily)', ha='center', fontsize=11,
        fontweight='bold', color='#1565C0')
draw_box(ax, 0.5, 7.5, 4, 1, 'User Interaction Data\n(clicks, bookings)', '#E3F2FD', '#1565C0')
draw_box(ax, 0.5, 5.8, 4, 1, 'Extract Search Sessions\n(sliding window, neg sampling)', '#BBDEFB', '#1565C0')
draw_box(ax, 0.5, 4.1, 4, 1, 'Train Listing2Vec\n(Skip-gram + improvements)', '#90CAF9', '#1565C0')
draw_arrow(ax, 2.5, 7.5, 2.5, 6.8)
draw_arrow(ax, 2.5, 5.8, 2.5, 5.1)

# --- Indexing Pipeline (middle) ---
ax.text(8, 9, 'INDEXING PIPELINE', ha='center', fontsize=11,
        fontweight='bold', color='#2E7D32')
draw_box(ax, 6, 7.5, 4, 1, 'Compute Embeddings\nfor ALL 5M listings', '#E8F5E9', '#2E7D32')
draw_box(ax, 6, 5.8, 4, 1, 'Build ANN Index\n(FAISS / ScaNN / HNSW)', '#C8E6C9', '#2E7D32')
draw_arrow(ax, 4.5, 4.6, 6, 8, '#666')
draw_arrow(ax, 8, 7.5, 8, 6.8)

# --- Prediction Pipeline (right) ---
ax.text(13.5, 9, 'PREDICTION PIPELINE', ha='center', fontsize=11,
        fontweight='bold', color='#BF360C')
draw_box(ax, 11.5, 7.5, 4, 1, '1. Embedding Fetcher\n(lookup or cold-start)', '#FFF3E0', '#FF9800')
draw_box(ax, 11.5, 5.8, 4, 1, '2. Nearest Neighbor\n(ANN search, top-K)', '#FFCCBC', '#BF360C')
draw_box(ax, 11.5, 4.1, 4, 1, '3. Re-Ranking\n(filters, diversity, rules)', '#E0F2F1', '#00695C')
draw_arrow(ax, 13.5, 7.5, 13.5, 6.8)
draw_arrow(ax, 13.5, 5.8, 13.5, 5.1)

# Index feeds into prediction
draw_arrow(ax, 10, 6.3, 11.5, 6.3, '#2E7D32')
draw_arrow(ax, 10, 8, 11.5, 8, '#2E7D32')

# User request and response
draw_box(ax, 11.5, 2.5, 4, 0.8, 'User Viewing Listing X', '#FFF9C4', '#F9A825', 10)
draw_arrow(ax, 13.5, 3.3, 13.5, 4.1, '#F9A825')

draw_box(ax, 11.5, 1.0, 4, 0.8, 'Show Similar Listings', '#FFF9C4', '#F9A825', 10)
draw_arrow(ax, 13.5, 4.1, 13.5, 1.8, '#00695C')

# Latency annotations
ax.text(15.7, 8, '~1ms', fontsize=9, color='#E65100', fontstyle='italic')
ax.text(15.7, 6.3, '~5ms', fontsize=9, color='#E65100', fontstyle='italic')
ax.text(15.7, 4.6, '~3ms', fontsize=9, color='#E65100', fontstyle='italic')

plt.tight_layout()
plt.show()

print('Key insight: Training and indexing happen OFFLINE (daily).')
print('Prediction happens ONLINE in ~10ms total.')
print('The expensive work (training, computing 5M embeddings) is done in batch.')

In [ ]:
# === Full Prediction Pipeline Simulation ===

import torch
import torch.nn as nn
from sklearn.metrics.pairwise import cosine_similarity

class SimilarListingServingSystem:
    """
    Complete serving system with all three prediction components.
    
    12-year-old version:
    When you look at a house on Airbnb, three things happen in 10ms:
    1. FETCH: Look up the house's secret code (embedding)
    2. SEARCH: Find houses with the most similar codes
    3. FILTER: Remove houses that do not match your preferences
    """
    def __init__(self, listings_df, embeddings):
        self.listings_df = listings_df
        self.embeddings = embeddings
        self.listing_ids = listings_df['listing_id'].tolist()
        self.id_to_idx = {lid: i for i, lid in enumerate(self.listing_ids)}
        self.reranker = ListingReRanker()
    
    def embedding_fetcher(self, listing_id):
        """
        Component 1: Embedding Fetcher
        If listing seen in training -> fetch from index
        If new listing -> cold start heuristic
        """
        if listing_id in self.id_to_idx:
            return self.embeddings[self.id_to_idx[listing_id]], 'cached'
        else:
            # Cold start: use average of same-city listings
            return np.mean(self.embeddings, axis=0), 'cold_start'
    
    def nearest_neighbor(self, query_embedding, top_k=50):
        """
        Component 2: Nearest Neighbor Search (ANN)
        In production: FAISS, ScaNN, or HNSW
        Here: brute force (acceptable for 5M with ANN)
        """
        similarities = cosine_similarity([query_embedding], self.embeddings)[0]
        top_indices = np.argsort(-similarities)[:top_k]
        
        candidates = []
        for idx in top_indices:
            row = self.listings_df.iloc[idx]
            candidates.append({
                'listing_id': row['listing_id'],
                'score': float(similarities[idx]),
                'city': row['city'],
                'price': row['price'],
                'listing_type': row['listing_type'],
                'host_id': row['host_id'],
                'available': row.get('available', True),
                'days_listed': row.get('days_listed', 100),
            })
        return candidates
    
    def recommend(self, listing_id, max_price=None, top_k=10):
        """
        Full prediction pipeline:
        Fetch embedding -> ANN search -> Re-rank
        """
        # Step 1: Fetch embedding
        embedding, source = self.embedding_fetcher(listing_id)
        
        # Step 2: Nearest neighbor search
        candidates = self.nearest_neighbor(embedding, top_k=50)
        
        # Remove the query listing itself
        candidates = [c for c in candidates if c['listing_id'] != listing_id]
        
        # Get target city
        if listing_id in self.id_to_idx:
            target_city = self.listings_df.iloc[self.id_to_idx[listing_id]]['city']
        else:
            target_city = candidates[0]['city'] if candidates else 'Unknown'
        
        # Step 3: Re-rank
        results = self.reranker.rerank(candidates, target_city, max_price, top_k)
        
        return results, source


# Create synthetic listing data
n_listings = 100
cities_pool = ['Miami'] * 40 + ['NYC'] * 30 + ['LA'] * 20 + ['Aspen'] * 10
types_pool = ['Beach House', 'Beach Condo', 'City Apt', 'Penthouse', 'Studio',
              'Villa', 'Cabin', 'Bungalow', 'Loft', 'Cottage']

listings_df = pd.DataFrame({
    'listing_id': [f'L{i}' for i in range(n_listings)],
    'city': [cities_pool[i % len(cities_pool)] for i in range(n_listings)],
    'listing_type': [types_pool[i % len(types_pool)] for i in range(n_listings)],
    'price': np.random.randint(80, 500, n_listings),
    'host_id': [f'host_{i % 30}' for i in range(n_listings)],
    'available': np.random.choice([True, True, True, False], n_listings),
    'days_listed': np.random.randint(1, 365, n_listings),
})

# Create embeddings (in production: from trained Listing2Vec)
embeddings = np.random.randn(n_listings, 32)
# Make same-city listings more similar
for city_offset, city in enumerate(['Miami', 'NYC', 'LA', 'Aspen']):
    mask = listings_df['city'] == city
    embeddings[mask] += np.random.randn(32) * 2  # City cluster
embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)

# Build system
system = SimilarListingServingSystem(listings_df, embeddings)

# Demo recommendation
query_listing = 'L0'
results, source = system.recommend(query_listing, max_price=350, top_k=8)

query_info = listings_df[listings_df['listing_id'] == query_listing].iloc[0]
print(f'=== Similar Listings to {query_listing} ===')
print(f'  Currently viewing: {query_info["listing_type"]} in {query_info["city"]}, ${query_info["price"]}/night')
print(f'  Embedding source: {source}')
print(f'  Max price filter: $350')
print()
print(f'{"Rank":<5} {"ID":<5} {"Score":<8} {"City":<8} {"Type":<15} {"Price":<8}')
print('-' * 55)
for i, r in enumerate(results, 1):
    print(f'  {i:<3} {r["listing_id"]:<5} {r["score"]:<8.3f} {r["city"]:<8} {r["listing_type"]:<15} ${r["price"]}')

## 4. Cold-Start for New Listings

### The New Kid in School

**Simple explanation**: When a brand new listing is added to Airbnb, nobody has clicked on it yet. It has no browsing history, so the model has no embedding for it. It is like a new kid at school who nobody knows yet. How do we include them?

### Three Strategies (in order of speed)

| Strategy | Speed | Quality | How It Works |
|----------|-------|---------|-------------|
| **Geographic proxy** | Instant | Decent | Use average embedding of same-neighborhood listings |
| **Feature-based init** | Instant | Better | Use listing attributes (price, type) to find similar known listings |
| **Daily retraining** | ~24 hours | Best | After 1 day of interaction data, retrain the model with the new listing |

In [ ]:
# === Cold-Start Strategy Comparison ===

class ColdStartHandler:
    """
    Handles new listings with no interaction data.
    
    12-year-old version:
    New kid at school? Here is how we help them make friends:
    1. GEOGRAPHIC: 'You live in the same neighborhood as Alex.
       You probably have similar interests!'
    2. FEATURE-BASED: 'You like skateboarding and pizza? So does
       Jordan! Let me introduce you.'
    3. DAILY RETRAIN: 'After a day, we have seen who you actually
       get along with, so now we know for real.'
    """
    def __init__(self, listings_df, embeddings):
        self.listings_df = listings_df
        self.embeddings = embeddings
    
    def geographic_proxy(self, city):
        """Average embedding of same-city listings."""
        mask = self.listings_df['city'] == city
        if mask.sum() == 0:
            return np.mean(self.embeddings, axis=0)
        city_embeddings = self.embeddings[mask.values]
        avg = np.mean(city_embeddings, axis=0)
        return avg / np.linalg.norm(avg)
    
    def feature_based(self, city, price, listing_type):
        """Find most similar listing by features and use its embedding."""
        df = self.listings_df.copy()
        # Score by feature similarity
        df['city_match'] = (df['city'] == city).astype(float) * 10
        df['type_match'] = (df['listing_type'] == listing_type).astype(float) * 5
        df['price_match'] = -abs(df['price'] - price) / 100
        df['feature_score'] = df['city_match'] + df['type_match'] + df['price_match']
        
        best_idx = df['feature_score'].idxmax()
        return self.embeddings[best_idx], self.listings_df.iloc[best_idx]['listing_id']


cold_handler = ColdStartHandler(listings_df, embeddings)

# Simulate a new listing
new_listing = {'city': 'Miami', 'price': 200, 'listing_type': 'Beach Condo'}

# Strategy 1: Geographic proxy
geo_emb = cold_handler.geographic_proxy(new_listing['city'])

# Strategy 2: Feature-based
feat_emb, proxy_id = cold_handler.feature_based(
    new_listing['city'], new_listing['price'], new_listing['listing_type']
)

# Compare: which strategy gives better neighbors?
geo_sims = cosine_similarity([geo_emb], embeddings)[0]
feat_sims = cosine_similarity([feat_emb], embeddings)[0]

print(f'=== Cold-Start for New Listing ===')
print(f'  New: {new_listing["listing_type"]} in {new_listing["city"]}, ${new_listing["price"]}/night')
print()

print('Strategy 1: Geographic Proxy (average of Miami listings)')
top5_geo = np.argsort(-geo_sims)[:5]
for i, idx in enumerate(top5_geo, 1):
    row = listings_df.iloc[idx]
    print(f'  #{i}: {row["listing_id"]} ({row["listing_type"]}, {row["city"]}, ${row["price"]}) sim={geo_sims[idx]:.3f}')

print(f'\nStrategy 2: Feature-Based (using proxy listing {proxy_id})')
top5_feat = np.argsort(-feat_sims)[:5]
for i, idx in enumerate(top5_feat, 1):
    row = listings_df.iloc[idx]
    print(f'  #{i}: {row["listing_id"]} ({row["listing_type"]}, {row["city"]}, ${row["price"]}) sim={feat_sims[idx]:.3f}')

print(f'\nStrategy 3: Daily Retraining')
print(f'  After ~24 hours, the listing accumulates enough clicks to be')
print(f'  included in the next training cycle. The model learns its true embedding.')

## 5. Online Learning and Model Updates

### The Daily Refresh

**Simple explanation**: Imagine if the librarian only learned about books from 2020 and never updated. By 2024, they would be recommending outdated books! The model needs to stay fresh.

**Staff-level detail**: The model is retrained daily on newly collected interaction data. This ensures:
- New listings get embeddings within 24 hours
- Changing user preferences are captured
- Seasonal patterns are reflected (beach houses in summer, ski cabins in winter)

### Retraining Strategy

| Approach | Description | Pros | Cons |
|----------|------------|------|------|
| **Full retrain** | Train from scratch on all data | Clean, reproducible | Slow (hours) |
| **Incremental update** | Fine-tune on new data only | Fast (minutes) | May drift |
| **Warm-start** | Initialize from previous model, train on all data | Best of both | Moderate time |

In [ ]:
# === Model Update Pipeline ===

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

# Timeline
days = ['Day 1', 'Day 2', 'Day 3', 'Day 4', 'Day 5']
for i, day in enumerate(days):
    x = 1 + i * 2.5
    
    # Day box
    draw_box(ax, x, 6.5, 2, 0.6, day, '#E3F2FD', '#1565C0', 10)
    
    # Collect data
    draw_box(ax, x, 5.2, 2, 0.6, 'Collect\nInteractions', '#C8E6C9', '#2E7D32', 8)
    draw_arrow(ax, x + 1, 6.5, x + 1, 5.8)
    
    # Train (overnight)
    draw_box(ax, x, 3.9, 2, 0.6, 'Retrain\nModel', '#FFCCBC', '#BF360C', 8)
    draw_arrow(ax, x + 1, 5.2, x + 1, 4.5)
    
    # Reindex
    draw_box(ax, x, 2.6, 2, 0.6, 'Recompute\nEmbeddings', '#F3E5F5', '#7B1FA2', 8)
    draw_arrow(ax, x + 1, 3.9, x + 1, 3.2)
    
    # Deploy
    draw_box(ax, x, 1.3, 2, 0.6, 'Deploy\nNew Index', '#E0F2F1', '#00695C', 8)
    draw_arrow(ax, x + 1, 2.6, x + 1, 1.9)
    
    # Arrow to next day
    if i < len(days) - 1:
        draw_arrow(ax, x + 2, 7.0, x + 2.5, 7.0, '#999')

# Time labels
ax.text(0.3, 5.5, '9am-9pm', fontsize=8, color='#666', fontstyle='italic')
ax.text(0.3, 4.2, '9pm-2am', fontsize=8, color='#666', fontstyle='italic')
ax.text(0.3, 2.9, '2am-5am', fontsize=8, color='#666', fontstyle='italic')
ax.text(0.3, 1.6, '5am-6am', fontsize=8, color='#666', fontstyle='italic')

ax.set_title('Daily Model Update Cycle', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('Daily cycle:')
print('  9am-9pm:  Collect user interaction data (clicks, bookings)')
print('  9pm-2am:  Retrain Listing2Vec on new data (warm-start from previous model)')
print('  2am-5am:  Recompute embeddings for all 5M listings')
print('  5am-6am:  Blue-green deploy: swap the ANN index atomically')
print('  6am:      Users see recommendations from the new model')
print('\nNew listings: get embeddings within 24 hours of first interactions')

## 6. A/B Testing for Listing Recommendations

### The Science Experiment

**Simple explanation**: Before changing the recommendation system for everyone, we test it on a small group. Half see the old recommendations, half see the new ones. After a week, we check: did the new group book more?

**Critical metrics for similar listings**:

| Metric | Definition | Why It Matters |
|--------|-----------|----------------|
| **CTR** | % who click a recommended listing | Engagement |
| **Session Book Rate** | % of sessions ending in booking | **Primary business metric** |
| **Listing Diversity** | Variety of listing types shown | User exploration |
| **Cold-Start Coverage** | % of new listings appearing in recs | Fairness to new hosts |

In [ ]:
# === A/B Testing for Similar Listings ===

from scipy import stats

np.random.seed(42)
n_users = 5000

# Control: basic Listing2Vec
control_ctr = np.random.beta(3, 7, n_users)
control_book_rate = np.random.beta(1, 19, n_users)  # ~5% book rate
control_diversity = np.random.uniform(0.3, 0.7, n_users)

# Treatment: Listing2Vec with hard negatives + global context
treatment_ctr = np.random.beta(3.3, 7, n_users)  # +5% CTR
treatment_book_rate = np.random.beta(1.2, 19, n_users)  # +15% book rate
treatment_diversity = np.random.uniform(0.35, 0.75, n_users)  # +5% diversity

# Run statistical tests
metrics = [
    ('Click-Through Rate', control_ctr, treatment_ctr),
    ('Session Book Rate', control_book_rate, treatment_book_rate),
    ('Listing Diversity', control_diversity, treatment_diversity),
]

print('=== A/B Test: Basic vs Improved Listing2Vec ===')
print(f'Users per group: {n_users:,}')
print(f'Duration: 2 weeks')
print()
print(f'{"Metric":<25} {"Control":>10} {"Treatment":>10} {"Lift":>8} {"p-value":>10} {"Sig?":>6}')
print('-' * 72)

for name, ctrl, treat in metrics:
    t_stat, p_val = stats.ttest_ind(ctrl, treat)
    lift = (np.mean(treat) - np.mean(ctrl)) / np.mean(ctrl) * 100
    sig = 'YES' if p_val < 0.05 else 'no'
    print(f'  {name:<23} {np.mean(ctrl):>10.4f} {np.mean(treat):>10.4f} '
          f'{lift:>+7.1f}% {p_val:>10.6f} {sig:>6}')

print(f'\nConclusion: Hard negatives + global context significantly improve')
print(f'BOTH engagement (CTR) AND the business metric (session book rate).')
print(f'Ship the improved model!')

In [ ]:
# === Visualize A/B Test Results ===

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, ctrl, treat) in zip(axes, metrics):
    # Compute means and CIs
    ctrl_mean = np.mean(ctrl)
    treat_mean = np.mean(treat)
    ctrl_ci = 1.96 * np.std(ctrl) / np.sqrt(len(ctrl))
    treat_ci = 1.96 * np.std(treat) / np.sqrt(len(treat))
    
    bars = ax.bar(['Control', 'Treatment'], [ctrl_mean, treat_mean],
                  yerr=[ctrl_ci, treat_ci], capsize=5,
                  color=['#90CAF9', '#EF5350'], edgecolor='black', linewidth=0.5)
    
    lift = (treat_mean - ctrl_mean) / ctrl_mean * 100
    ax.set_title(f'{name}\n(+{lift:.1f}% lift)', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + ctrl_ci + 0.002,
                f'{bar.get_height():.4f}', ha='center', fontsize=9)

plt.suptitle('A/B Test Results: Basic vs Improved Listing2Vec', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Interview Cheat Sheet: Ranking and Serving

### Key Phrases for the Interview

| Topic | Key Phrase |
|-------|-----------|
| Re-ranking | "We apply deterministic business rules AFTER embedding-based retrieval: price filters, availability, host diversity, geographic constraints" |
| Position bias | "Users click higher positions regardless of relevance. We correct with inverse propensity weighting or position-aware training" |
| Cold start | "Geographic proxy for instant recommendations; feature-based initialization for better quality; daily retraining for true embeddings within 24 hours" |
| Serving | "Three pipelines: training (daily), indexing (pre-compute 5M embeddings), prediction (fetch -> ANN -> re-rank in ~10ms)" |
| A/B testing | "Randomize at user level, run 2 weeks, primary metric = session book rate, guardrail = CTR" |
| Scaling | "ANN search (FAISS/ScaNN) searches 5M vectors in <10ms; embeddings are pre-computed offline" |

### Common Follow-Up Questions

1. **"How would you handle seasonality?"** -- Train separate seasonal models or add a time feature. Beach houses should rank higher in summer, ski cabins in winter.

2. **"How would you personalize for logged-in users?"** -- Incorporate the user's past booking embeddings as additional context. If they booked luxury villas before, boost luxury results.

3. **"What if a host gaming the system by creating many similar listings?"** -- Host diversity rules (max 2 per host) plus anomaly detection for suspicious listing patterns.

4. **"How would you measure fairness for new hosts?"** -- Track cold-start coverage: what % of new listings appear in recommendations within their first week?

## Key Takeaways

1. **Re-ranking is essential**: Raw embedding similarity is not enough. Business rules (price, availability, diversity) must be applied post-retrieval.

2. **Position bias contaminates training data**: Users click higher positions regardless of quality. Correct with IPW or position-aware features.

3. **Three-pipeline serving**: Training (daily offline) + Indexing (pre-compute embeddings) + Prediction (real-time in ~10ms). The expensive work is done offline.

4. **Cold-start is solvable**: Geographic proxy for instant results, feature-based for better quality, daily retraining for true embeddings within 24 hours.

5. **Daily retraining keeps the model fresh**: New listings get embeddings, seasonal patterns are captured, and user behavior shifts are reflected.

6. **A/B test everything**: Session book rate is the primary business metric. CTR alone is insufficient because clicks are not bookings.

7. **ANN search makes it fast**: FAISS/ScaNN searches 5M vectors in <10ms -- critical for real-time serving.

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)